In [1]:
%load_ext autoreload
%autoreload 2

In [153]:
import os
import sys
import json
from pathlib import Path
from collections import defaultdict
import re

os.chdir("/home/kaariaa3/mscthesis/")
sys.path.append("./src/")  # Add module directory to path

import pandas as pd
import numpy as np
import statsmodels.stats.contingency_tables as sm

from utils.plots import calculate_metrics, calculate_accuracy
from utils.tools import get_config, collect_jobs, prettify_table, format_table, aggregate_results
from utils.constants import PRED_COLS

In [90]:
def format_model_name(name):
    parts = name.split("-")
    match = re.search(r"\d{1,2}B", name)
    return f"{parts[0]}-{match.group(0)}" if match else name

def get_jobs_per_model_and_suite(jobs):
    
    print(f"Processing {len(jobs)} jobs.")
    # Dict of model -> suite -> seed -> jobId
    table = defaultdict(lambda: defaultdict(lambda: defaultdict(str)))
    for job in jobs:
        config = get_config(job["config_json"])
        model = config.get("model").split("/")[1]
        suite = get_suite(config)
        seed = config["seed"]

        table[model][suite][seed] = config.get("jobid")

    return table

def get_suite(cfg):

    #print(cfg)
    
    n_demos = cfg["number_of_demonstrations"]

    type_demos = (
        "non" 
        if n_demos == 0
        else "neg" 
        if cfg["type_of_demonstrations"] == -1
        else "mix" 
        if cfg["type_of_demonstrations"] == 0
        else "pos"
    )
    instr = (
        "impl" 
        if cfg["use_instructions"] == 0
        else "expl"
    )

    return f"{n_demos}-{type_demos}-{instr}"

In [91]:
models = ["Qwen3-32B", "Qwen3-8B", "Mistral-7B-Instruct-v0.3", "Mistral-Small-3.2-24B-Instruct-2506", "Llama-3.1-8B-Instruct", "Llama-3.3-70B-Instruct"]

model_map = {
    format_model_name(model): model for model in models
}

In [81]:
# Collect raw data
basepath = "./outputs/v6"
finished_jobs = collect_jobs(basepath)
jobs_list = [job for _, job_list in finished_jobs.items() for job in job_list] # Flatten

jobs_grouped = get_jobs_per_model_and_suite(jobs_list)

Processing 330 jobs.


In [83]:
#jobs_grouped

In [176]:
def get_paired_predictions(jobs, model, baseline, against):

    def _align_dataframes(df1, df2, key_cols=("title", "The exercise description matched the selected theme (Yes/No)", "The exercise description matched the selected topic (Yes/No)", "Included concepts that were too advanced (Yes/No)")):
        # Inner join on key columns to find common rows
        common_keys = pd.merge(
            df1[list(key_cols)].drop_duplicates(),
            df2[list(key_cols)].drop_duplicates(),
            on=list(key_cols),
            how="inner"
        )
    
        # Keep only matching rows
        df1_aligned = df1.merge(common_keys, on=list(key_cols), how="inner")
        df2_aligned = df2.merge(common_keys, on=list(key_cols), how="inner")
    
        # Sort identically so row order matches
        df1_aligned = df1_aligned.sort_values(list(key_cols)).reset_index(drop=True)
        df2_aligned = df2_aligned.sort_values(list(key_cols)).reset_index(drop=True)

        return df1_aligned, df2_aligned

    def _get_model_family(model):
        match model.split("-")[0]:
            case "Qwen3":
                return "Qwen"
            case "Mistral":
                return "mistralai"
            case "Llama":
                return "meta-llama"
            case _:
                return "default"

    def _get_csv(model, jobid):
        fam = _get_model_family(model)
        return f"./outputs/v6/{fam}/{model}/{jobid}/result.csv"

    def _get_errored_rows(df):
        return ~df[PRED_COLS].isin(["\"yes\"", "\"no\""]).all(axis=1)

    # Mapping seed -> jobid
    jobids_base = jobs[model][baseline]
    jobids_against = jobs[model][against]

    # Append all predictions to these dataframes for pairwise calculations
    preds_base = None
    preds_against = None   

    # Calculate pairwise differences
    for seed in jobids_base.keys():
        jobid_bl = jobids_base[seed]
        jobid_ag = jobids_against[seed]
      
        base = pd.read_csv(_get_csv(model, jobid_bl), sep=";")       
        comp = pd.read_csv(_get_csv(model, jobid_ag), sep=";")

        # We need to drop the rows that were used as demonstrations from the baseline,
        # Because we can't calculate the pairwise connections for these rows
        base, comp = _align_dataframes(dfbl, dfag)

        # Since both dataframes may contain errors and on different samples,
        # need to remove erroneous samples from both dataframes
        error_mask = (_get_errored_rows(base) | _get_errored_rows(comp))
        base = base.loc[~error_mask]
        comp = comp.loc[~error_mask]

        # Append predictions
        if preds_base is None:
            preds_base = base[PRED_COLS].copy()
            preds_against = comp[PRED_COLS].copy()
        else:
            preds_base = pd.concat([preds_base, base[PRED_COLS]])
            preds_against = pd.concat([preds_against, comp[PRED_COLS]])

    # All predictions have been collected into two dataframes in correct order
    return preds_base.reset_index(drop=True), preds_against.reset_index(drop=True)

bl = "0-non-expl"
ag = "6-neg-impl"

b, a = get_paired_predictions(jobs_grouped, model_map["Mistral-24B"], bl, ag)

In [173]:
def build_contingency_tables(df1, df2):
    return [pd.crosstab(df1[col], df2[col]) for col in PRED_COLS]
    
tabs = build_contingency_tables(b, a)

tabs[0]

themeCorrect,"""no""","""yes"""
themeCorrect,,
"""no""",820,245
"""yes""",355,1195


In [177]:
def my_mcnemar(tables, e=True):
    return sm.mcnemar(tables[0], exact=e)

print(my_mcnemar(tabs, False))

pvalue      8.590773541559997e-06
statistic   19.801666666666666


In [171]:
#dfbl = pd.read_csv("outputs/v6/mistralai/Mistral-Small-3.2-24B-Instruct-2506/17053874/result.csv", sep=";")
#dfag = pd.read_csv("outputs/v6/mistralai/Mistral-Small-3.2-24B-Instruct-2506/17053902/result.csv", sep=";")